# CognitiveMirage — Kaggle Benchmarks Submission
**Track:** Metacognition  
**Author:** Dayeon Kang  
**Benchmark version:** v3 (50 tasks · 5 families · LLM-as-judge)

---

## Overview

This notebook implements and documents the CognitiveMirage benchmark, which tests **metacognitive monitoring** — a model's ability to detect when it is about to be wrong, before it answers.

**Central finding:** Global TDR–accuracy correlation r = −0.94. The two most accurate models (claude-opus-4-5 at 100%, gpt-4o at 98%) rank last on metacognitive monitoring. Capability and metacognitive awareness are not just separable — they are negatively correlated.

In [ ]:
import json
import math
from pathlib import Path

# ── Load pre-computed results ──────────────────────────────────────────────
RESULTS_FILE = Path('v3_analysis.json')
with open(RESULTS_FILE) as f:
    results = json.load(f)

profiles = results['profiles']
leaderboard = results['leaderboard']
global_corr = results['global_correlation']
family_corr = results['family_correlations']

MODEL_ORDER = [name for name, _ in leaderboard]
print(f'Loaded results for {len(profiles)} models: {MODEL_ORDER}')

## 1. Benchmark Design

### Task Families & Scoring Modes

| Family | Trap Type | Scoring Mode | n tasks |
|---|---|---|---|
| `expertise_trap` | Domain knowledge invites overconfident answers | `expertise_inverted` | 8 |
| `forced_abstention` | Task is genuinely unanswerable; abstention is correct | `abstain_binary` | 12 |
| `confidence_inversion` | Answer is easy; confidence should be *lowered* by context | `rubric` | 10 |
| `over_specification` | Excess constraints make question vacuous | `rubric` | 8 |
| `control_baseline` | Clean answerable tasks; calibration baseline | `rubric` | 12 |

Each mirage task has a matched clean variant. **Trap Detection Rate (TDR)** measures whether the model flags the flaw before answering.

**Metacognitive Index (MI) = TDR × 0.5 + max(0, CalibΔ) × 0.5**

In [ ]:
# ── Display leaderboard ────────────────────────────────────────────────────
print(f"{'Rank':<5} {'Model':<22} {'MI':<8} {'TDR':>7} {'CleanAcc':>9} {'CalibΔ':>8}")
print('-' * 60)
for rank, (model, mi) in enumerate(leaderboard, 1):
    p = profiles[model]
    print(f"{rank:<5} {model:<22} {mi:<8.4f} {p['tdr_global']:>7.1%} {p['aq_clean']:>9.1%} {p['calib_delta']:>+8.3f}")

## 2. Key Finding — The Sign-Flip

The global TDR–accuracy correlation is strongly negative: models that perform best on factual tasks perform worst on metacognitive monitoring. This is not an artefact of a single model — it is LOO-stable and statistically robust.

The sign **reverses** within the `confidence_inversion` family (r = +0.97), where stronger models correctly lower their confidence when context demands it. The flip is family-dependent and theoretically meaningful.

In [ ]:
# ── Fisher z-transform 95% CI for Pearson r ───────────────────────────────
def fisher_ci(r, n, alpha=0.05):
    """Return (point, lo, hi) via Fisher z-transform."""
    z = math.atanh(r)
    se = 1.0 / math.sqrt(n - 3)
    z_crit = 1.96  # 95% two-sided
    lo = math.tanh(z - z_crit * se)
    hi = math.tanh(z + z_crit * se)
    return r, lo, hi


# ── Compute and display CIs ────────────────────────────────────────────────
n = 6  # number of models

r_global = global_corr['tdr_vs_accuracy']['r']
_, lo, hi = fisher_ci(r_global, n)
print(f"Global TDR vs accuracy:  r = {r_global:+.4f}  95% CI [{lo:+.3f}, {hi:+.3f}]")
excludes = (lo < 0 < hi) is False
print(f"  CI excludes zero: {excludes}")
print()

print(f"{'Family':<25} {'r':>7}  {'95% CI':>22}  {'Excludes 0?':>12}")
print('-' * 70)
for fam, fc in family_corr.items():
    r = fc['r']
    if r == 0.0:
        print(f"{fam:<25} {r:>7.4f}  {'n/a (r=0)':>22}  {'—':>12}")
        continue
    _, lo, hi = fisher_ci(r, n)
    excl = 'YES ***' if (lo > 0 and hi > 0) or (lo < 0 and hi < 0) else 'no'
    print(f"{fam:<25} {r:>7.4f}  [{lo:>+.3f}, {hi:>+.3f}]  {excl:>12}")

In [ ]:
# ── LOO stability for global r ─────────────────────────────────────────────
import statistics

xs = [profiles[m]['aq_clean'] for m in MODEL_ORDER]
ys = [profiles[m]['tdr_global'] for m in MODEL_ORDER]

def pearson_r(x, y):
    n = len(x)
    mx, my = sum(x)/n, sum(y)/n
    num = sum((xi - mx)*(yi - my) for xi, yi in zip(x, y))
    den = math.sqrt(sum((xi - mx)**2 for xi in x) * sum((yi - my)**2 for yi in y))
    return num / den if den > 0 else 0.0

loo_rs = []
for i in range(len(MODEL_ORDER)):
    xs_loo = [v for j, v in enumerate(xs) if j != i]
    ys_loo = [v for j, v in enumerate(ys) if j != i]
    loo_rs.append(pearson_r(xs_loo, ys_loo))

print("LOO stability — global TDR vs accuracy r (removing one model at a time):")
for model, r_loo in zip(MODEL_ORDER, loo_rs):
    print(f"  leave out {model:<22}: r = {r_loo:+.4f}")
print(f"\n  Range: [{min(loo_rs):+.4f}, {max(loo_rs):+.4f}]")
print(f"  Sign stable: {all(r < 0 for r in loo_rs)}")
print(f"  Min |r|: {min(abs(r) for r in loo_rs):.4f}")

In [ ]:
# ── Effect size: clean vs mirage task scores ──────────────────────────────
# Cohen's d = 2.65 (pre-computed from per-task score distributions,
# stored in v3_analysis.json['effect_size_clean_vs_mirage'])
# Using per-task scores (n=50 each) gives the correct effect size.
# Computing on 6 per-model averages would inflate d artificially.
cohen_d = results['effect_size_clean_vs_mirage']['cohens_d']

print(f"Cohen's d (clean vs mirage, per-task scores): {cohen_d:.3f}")
print(f"  Interpretation: {'large (d > 0.8)' if cohen_d > 0.8 else 'medium' if cohen_d > 0.5 else 'small'}")
print(f"  Mirage tasks are substantially harder, confirming the traps are non-trivial.")

## 3. Per-Family TDR Analysis

In [ ]:
# ── Per-family TDR breakdown ───────────────────────────────────────────────
families = ['confidence_inversion', 'forced_abstention', 'expertise_trap',
            'over_specification', 'control_baseline']

header = f"{'Model':<22}" + "".join(f"{f[:9]:>12}" for f in families)
print(header)
print('-' * (22 + 12 * len(families)))

for model in MODEL_ORDER:
    p = profiles[model]
    row = f"{model:<22}"
    for fam in families:
        tdr = p['family'][fam]['tdr']
        row += f"{tdr:>12.1%}"
    print(row)

In [ ]:
# ── Per-family scatter: TDR vs Clean Accuracy ──────────────────────────
# Visualise the per-family TDR–accuracy relationship.
# Each subplot shows one family; points are models (x=CleanAcc, y=TDR).
# The family-level r (from family_corr) is annotated on each panel.
import matplotlib.pyplot as plt

FAMILY_COLORS = {
    'confidence_inversion': '#2196F3',
    'forced_abstention':    '#F44336',
    'expertise_trap':       '#FF9800',
    'over_specification':   '#9C27B0',
    'control_baseline':     '#9E9E9E',
}
FAMILY_LABELS = {
    'confidence_inversion': 'Confidence Inversion',
    'forced_abstention':    'Forced Abstention',
    'expertise_trap':       'Expertise Trap',
    'over_specification':   'Over-specification',
    'control_baseline':     'Control Baseline',
}
SHORT_NAME = {m: m.replace('claude-', 'cl-').replace('gpt-', 'g-').replace('-preview', '') for m in MODEL_ORDER}

families = ['confidence_inversion', 'forced_abstention', 'expertise_trap',
            'over_specification', 'control_baseline']

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes_flat = axes.flatten()

for ax_i, fam in enumerate(families):
    ax = axes_flat[ax_i]
    r_val = family_corr[fam]['r']
    color = FAMILY_COLORS[fam]

    xs = [profiles[m]['aq_clean'] for m in MODEL_ORDER]
    ys = [profiles[m]['family'][fam]['tdr'] for m in MODEL_ORDER]

    ax.scatter(xs, ys, color=color, s=90, zorder=3, edgecolors='white', linewidths=0.8)
    for i, m in enumerate(MODEL_ORDER):
        ax.annotate(SHORT_NAME[m], (xs[i], ys[i]), xytext=(5, 3),
                    textcoords='offset points', fontsize=7, color='#333333')

    # OLS regression line (computed with stdlib, no numpy needed)
    n_pts = len(xs)
    xmean = sum(xs) / n_pts
    ymean = sum(ys) / n_pts
    ssxy = sum((xi - xmean) * (yi - ymean) for xi, yi in zip(xs, ys))
    ssx  = sum((xi - xmean) ** 2 for xi in xs)
    if ssx > 1e-10:
        slope = ssxy / ssx
        intercept = ymean - slope * xmean
        x0, x1 = min(xs) - 0.04, max(xs) + 0.04
        ax.plot([x0, x1], [slope * x0 + intercept, slope * x1 + intercept],
                color=color, alpha=0.55, linewidth=1.8, linestyle='--')

    excl = ''
    if r_val != 0.0:
        _, lo_f, hi_f = fisher_ci(r_val, n_pts)
        if (lo_f > 0 and hi_f > 0) or (lo_f < 0 and hi_f < 0):
            excl = ' ***'

    ax.set_title(f"{FAMILY_LABELS[fam]}\nr = {r_val:+.3f}{excl}",
                 fontsize=9.5, fontweight='bold', color='#222222')
    ax.set_xlabel('Clean Accuracy', fontsize=8)
    ax.set_ylabel('TDR', fontsize=8)
    ax.set_xlim(-0.05, 1.08)
    ax.set_ylim(-0.08, 1.10)
    ax.tick_params(labelsize=7)
    ax.grid(True, alpha=0.25)
    ax.axhline(0.5, color='gray', linewidth=0.4, linestyle=':')
    ax.axvline(0.5, color='gray', linewidth=0.4, linestyle=':')

axes_flat[-1].set_visible(False)
fig.suptitle('Per-Family: TDR vs Clean Accuracy (n=6 models each)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('per_family_scatter.png', dpi=100, bbox_inches='tight')
plt.show()
print('Figure saved: per_family_scatter.png')
print('(*** = 95% Fisher CI excludes zero)')

## 4. Kaggle Benchmarks SDK Integration

The `kaggle_task.py` file implements the Kaggle Benchmarks SDK interface. It loads all 50 tasks from `v3_tasks_50.json` and supports three scoring modes.

To run locally (without Kaggle SDK):

In [ ]:
# ── Verify benchmark tasks load correctly ─────────────────────────────────
tasks_file = Path('v3_tasks_50.json')
with open(tasks_file) as f:
    tasks = json.load(f)

family_counts = {}
for t in tasks:
    fam = t.get('family', 'unknown')
    family_counts[fam] = family_counts.get(fam, 0) + 1

print(f"Total tasks: {len(tasks)}")
print("Family breakdown:")
for fam, count in sorted(family_counts.items()):
    print(f"  {fam:<30}: {count} tasks")

# Spot check a task
sample = tasks[0]
print(f"\nSample task ({sample['family']} / {'MIRAGE' if sample.get('is_mirage') else 'CLEAN'}):")
print(f"  ID: {sample['id'][:12]}...")
print(f"  Scoring mode: {sample['scoring_mode']}")
print(f"  Prompt (first 120 chars): {sample['prompt'][:120]}...")

## 5. Conclusions

### The Sign-Flip Summary

| Metric | Value | 95% CI | Stable? |
|---|---|---|---|
| Global TDR vs accuracy | r = −0.94 | [−0.99, −0.56] | LOO-stable, sign=negative across all 6 LOO folds |
| confidence_inversion | r = +0.97 | [+0.75, +1.00] | CI excludes zero |
| forced_abstention | r = −0.92 | [−0.99, −0.42] | CI excludes zero |
| Cohen's d (clean vs mirage) | d = 2.65 | — | Large effect |

### Interpretation

The sign-flip is **family-dependent** and **theoretically grounded**:

1. **Forced abstention** (r = −0.92): A capable model recognizes the task genre (Q&A) and answers fluently, even when the task is unanswerable. Less capable models hedge. Capability is a liability here.

2. **Confidence inversion** (r = +0.97): The trap requires noticing that contextual cues should *lower* confidence. Stronger models pick up on these cues; weaker models don't. Capability helps here.

3. **Global** (r = −0.94): The forced_abstention family dominates the global signal, producing the net negative correlation.

**Key claim:** Metacognitive monitoring and factual competence are dissociable cognitive faculties. A model can be highly accurate on clean tasks while systematically failing to detect traps. No existing benchmark measures this directly — CognitiveMirage fills this gap.

### Files

| File | Purpose |
|---|---|
| `v3_tasks_50.json` | 50 benchmark tasks (v3) |
| `v3_judge_evaluator.py` | LLM-as-judge evaluation engine |
| `v3_statistical_analysis.py` | Cross-model analysis, correlations, LOO, CI |
| `v3_analysis.json` | Full results from 6-model evaluation |
| `kaggle_task.py` | Kaggle Benchmarks SDK wrapper |
| `dashboard.html` | Self-contained interactive results dashboard |
| `requirements.txt` | Python dependencies |